# OFDM и Resource Grid

В основе физического уровня 5G NR лежит технология мультиплексирования с ортогональным частотным разделением каналов - CP-OFDM. 5G требует большой гибкости для обслуживания совершенно разных сценариев (от IoT с маленькими пакетами до потокового 4K-видео).

Для управления ресурсами используется **Resource Grid**. Всю доступную пропускную способность можно представить в виде двумерной матрицы, где:
*   Ось X - **время** - разделена на **Frames**, **Subframes**, **Slots** и **OFDM-символы**;
*   Ось Y - **частота** - разделена на **Subcarriers**.

Базовая станция на физическом уровне распределяет данные абонентов по ячейкам этой сетки, после чего передает их на обработку помехоустойчивым кодекам и далее через системную шину AXI в память. Именно размер и заполненность этой сетки определяют вычислительную нагрузку на аппаратную часть.

Структура Resource Grid состоит из блоков:
1.  **Resource Element** - самая маленькая и базовая единица сетки. Она равна 1 поднесущей по частоте на 1 OFDM-символ по времени. Один Resource Element может переносить комплексное число, представляющее модулированный символ (например, QPSK, 16-QAM, 256-QAM).
2.  **Resource Block** - в 5G ресурсный блок определяется только в частотной области и состоит из **12** последовательных поднесущих.
3.  **Нумерология - Numerology** - особенность 5G. В LTE расстояние между поднесущими всегда было 15 кГц. В 5G этот параметр гибкий: 15 кГц, 30 кГц, 60 кГц, 120 кГц и т.д. Чем шире поднесущая т.е. выше нумерология, тем короче OFDM-символ по времени. Это напрямую влияет на то, с какой скоростью кодеки базовой станции должны переваривать данные.

### [Sqimway](https://www.sqimway.com/lte_resource_grid.html)
![Resource Grid](images/sqimway.png)

В Resource Grid мы видим обозначения Resource Element

![Resource Element](images/res_elem.png)

1.  **PSCH / SSCH (Зеленый и светло-зеленый):** Сигналы синхронизации. С их помощью устройство понимает, что это сеть, и подстраивается под её ритм.
2.  **PBCH (Оранжевый):** Вещательный канал. Передает самую важную информацию о соте (MIB), без которой устройство не сможет подключиться.
3.  **RS (Красный):** **Reference Signals (Опорные сигналы).** Самые важные точки на сетке. По ним устройство оценивает качество связи. Если их много, связь стабильнее, но места для данных меньше.
4.  **PCFICH (Синий):** Канал, который сообщает устройству значение CFI (сколько символов занято управлением).
5.  **PHICH (Розовый):** Канал подтверждения приема (Hybrid-ARQ).
6.  **PDCCH (Желтый):** **Тот самый контрольный канал** В нем лежат команды сколько выделено ресурсов абоненту.
7.  **PDSCH (Белый/Прозрачный):** **Полезная нагрузка.** Сюда попадают пакеты по условно 100, 30, 10 байт.
8.  **Черный / Серый:** Неиспользуемые или зарезервированные ресурсы.

Параметры для настройки:

![sqimway parameters](images/sqimway_firstparam.png)

*   **Duplexing mode:** способ разделения приема и передачи.  Означает, что для загрузки (DL) и отдачи (UL) используются разные частоты.
*   **Channel bandwidth (1.4 MHz):** ширина канала. Высота сетки. 1.4 МГц — это самый узкий канал в LTE, он состоит из **6 ресурсных блоков (PRB)** или **72 поднесущих**. Чем шире канал, тем больше данных можно запихнуть.
*   **Cyclic prefix (Normal):** защитный интервал между символами, чтобы они не накладывались друг на друга из-за эха. В режиме Normal в одном слоте помещается **7 символов**.
*   **Tx antenna port (0 / 2):** количество антенн. От этого зависит, сколько будет красных квадратов (Reference Signals), так как каждой антенне нужны свои проверочные сигналы.
*   **Physical layer cell ID (0):** уникальный номер соты. От него зависит узор расположения сигналов синхронизации и проверочных сигналов, чтобы соседние вышки не светили в одни и те же ячейки.
*   **Control format indicator (CFI - 1):** он говорит, сколько первых символов в каждом подкадре отдано под управление (PDCCH). CFI=1 значит, что только первый символ занят командами, а остальные — данными.
*   **PHICH Ng factor / duration:** параметры канала, который подтверждает получение данных от устройствва (ACK/NACK). 

# py3gpp

Реализуем [калькулятор](calculator.py) для расчета нагрузки на железо на базе py3gg
(Для запуска калькулятора необходимо установить [py3gpp](https://github.com/catkira/py3gpp) )

(Полный код реализован в отдельном файле .py)
Разберем блоки кода:

In [ ]:
## Настройка несущей
carrier = py3gpp.nrCarrierConfig()
carrier.NSizeGrid = 273        # Ширина сетки
carrier.SubcarrierSpacing = 30 # Нумерология

Создает саму сетку (как и параметр Channel bandwidth в Sqimway). Нумерология выбрана 30 кГц, как самая распространенная(из 15 30 60 120) Ширина определяются по 3GPP (TS 38.101). Для полосы 100 МГц и нумерологии максимальное кол-во Resource Block равно 273. 

In [ ]:
## Планировщик
allocated_prb = max(1, math.ceil(packet_size_bytes / 10))
pdsch.PRBSet = list(range(allocated_prb))

Определяет сколько Resource Elements дать пакету. 

In [ ]:
    ## определение кодека (LDPC BG1 BG2)
    # по стандарту 3GPP если Transport Block Size - т.е. размер данных - маленький то BG2 если большой то BG1.
    codec_type = "LDPC Base Graph 2"
    if packet_size_bytes > 479:
        codec_type = "LDPC Base Graph 1"

Определяет какие будут нагружаться кодеки в зависимости от размера данных

In [ ]:
## Mapping 
result_indices = py3gpp.nrPDSCHIndices(carrier, pdsch)
num_re = len(ind)

Вычисление положение Resource Elements в сетке

In [ ]:
    ## трафик для шины AXI
    axi_bus_bytes = num_re * 4

Каждый Resource Elements обычно представляется комплексным числом I/Q.
Если I и Q это 16-битные числа (2 байта), то 1 RE = 4 байта сырых данных.
Для определения сколько байт берет блок и что пойдет через шину AXI.

Калькулятор выводит массив с параметрами (пример для пакета 100 байт)
- **Packet Size: 100 байт** - входная нагрузка от пользователя
- **Allocated PRBs: 10** - выделенные ресурсные блоки (100 байт делим на 10 = 10 блоков, при этом меньше одного блока выделить невозможно, по этому что на 10 байт, что на 1 байт будет выделен 1 блок, соответственно 1 байт передавать дороже чем 10 :) )
- **Resource Elements Used: 1080** - общее кол-во занятых Resource Elements (10 PRB * 12 Resource Elements. Пакет занимает 9 символов во времени<sup>1</sup>, т.е. 120*9=1080)
- **Codec: LDPC Base Graph 2** - задаем порог для кодека, [ссылка на справки по выбору базового графа LDPC](https://sharetechnote.com/html/5G/5G_PDSCH.html) (п.2)<sup>2</sup>
- **AXI Bus Load: 4320 байт сырых I/Q данных** - Один Resource Element в памяти представлена как комплексное число: 16 бит для части I и 16 бит для части Q. Итого 4 байта на один Resource Element. Нужно чтобы понять выдержит ли AXI. Если шина AXI медленная, она не успеет прокачать эти 4320 байта за время одного слота (0.5 мс), и связь оборвется.

**Примечания**

<sup>1</sup> Почему из 14 возможных символов используем 9? (всего OFDM от 0 до 13) Отнимаем 1,2 или 3 символа на контроль (у нас 1 и 2 зарезервированы). 11, 12, 13 остаются пустыми для успешного переключения с передачи на прием. Мы задем pdsch.SymbolAllocation = [2, 9] - 2 - номер стартового символа, 9 - кол-во символов (2+9=11)


<sup>2</sup> Выбор базового графа определяется размером транспортного блока (A) и скоростью кодирования (R) в соответствии со следующими критериями. (Кратко говоря BS1 для больших пакетов, BS2 для малых пакетов)

$$
\begin{align*}
A \le 292 & \rightarrow \text{LDPC base graph 2} \\
A \le 3824 \text{ and } R \le 0.67 & \rightarrow \text{LDPC base graph 2} \\
R \le 0.25 & \rightarrow \text{LDPC base graph 2} \\
\text{else} & \rightarrow \text{LDPC base graph 1}
\end{align*}
$$